# Day 2 — Logit Lens

Project each layer's hidden state through `lm_head` to see what the model predicts at every layer.

**Key question:** At which layer does the correct answer first appear?

- `embed` = raw token embeddings (no transformer layers yet)
- `L1–L16` = output after each transformer layer
- We apply `model.model.norm` + `model.lm_head` to each state to get probabilities

In [ ]:
import sys
sys.path.insert(0, '..')

from src.inspector.model_loader import load_model
from src.inspector.logit_lens import (
    run_logit_lens,
    print_table,
    plot_heatmap,
    find_emergence_layer,
    run_all_prompts,
)
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

model, tokenizer = load_model()

## Part 1 — France capital: when does 'Paris' emerge?

In [ ]:
result_france = run_logit_lens(model, tokenizer, "The capital of France is")
print_table(result_france)

layer = find_emergence_layer(result_france, "Paris")
print(f"'Paris' first appears at layer: {layer} of 16")

In [ ]:
plot_heatmap(result_france, save_path=Path("../outputs/logit_lens/nb_france.png"))
img = mpimg.imread("../outputs/logit_lens/nb_france.png")
plt.figure(figsize=(15, 5))
plt.imshow(img)
plt.axis('off')
plt.show()

## Part 2 — Math: does '4' appear for '2 + 2 ='?

In [ ]:
result_math = run_logit_lens(model, tokenizer, "2 + 2 =")
print_table(result_math)

layer = find_emergence_layer(result_math, "4")
if layer == -1:
    print("'4' never appeared in top-5 — 1B model can't do arithmetic")
else:
    print(f"'4' first appears at layer: {layer}")

In [ ]:
plot_heatmap(result_math, save_path=Path("../outputs/logit_lens/nb_math.png"))
img = mpimg.imread("../outputs/logit_lens/nb_math.png")
plt.figure(figsize=(15, 5))
plt.imshow(img)
plt.axis('off')
plt.show()

## Part 3 — Water formula: 'WATER' → 'H' transition

In [ ]:
result_water = run_logit_lens(model, tokenizer, "The chemical formula for water is")
print_table(result_water)

layer = find_emergence_layer(result_water, "H")
print(f"'H' first appears at layer: {layer} of 16")
print("(Note: model predicts 'WATER' at L12-13 before switching to 'H' at L14)")

In [ ]:
plot_heatmap(result_water, save_path=Path("../outputs/logit_lens/nb_water.png"))
img = mpimg.imread("../outputs/logit_lens/nb_water.png")
plt.figure(figsize=(15, 5))
plt.imshow(img)
plt.axis('off')
plt.show()

## Part 4 — Emergence summary across all key experiments

In [ ]:
experiments = [
    ("The capital of France is", "Paris"),
    ("2 + 2 =", "4"),
    ("The chemical formula for water is", "H"),
    ("The cat sat on the", "mat"),
    ("def hello_world():\n    print(", "Hello"),
    ("Roses are red, violets are", "blue"),
]

print(f"{'Prompt':<45} {'Target':<10} {'Layer'}")
print("-" * 65)
for prompt, target in experiments:
    result = run_logit_lens(model, tokenizer, prompt)
    layer = find_emergence_layer(result, target)
    label = "never" if layer == -1 else ("embed" if layer == 0 else f"L{layer}")
    short_prompt = repr(prompt[:42])
    print(f"{short_prompt:<45} {repr(target):<10} {label}")

## Part 5 — Full test suite (all categories)

In [ ]:
# Runs all 13 prompts from data/test_prompts.json and saves heatmaps
all_results = run_all_prompts(model, tokenizer, prompts_path="../data/test_prompts.json", print_tables=False)
print(f"\nCompleted {sum(len(v) for v in all_results.values())} prompts across {len(all_results)} categories")